# Model Training

## Objective

Train machine learning models to predict whether a patient will miss a scheduled healthcare appointment.

The training process will include:

- Loading the processed dataset
- Separating features and target
- Train/test split
- Identifying numerical and categorical features
- Building a preprocessing pipeline
- Training a Logistic Regression baseline
- Evaluating the baseline model

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# Load processed data

In [3]:
df = pd.read_csv(
    "../data/processed/healthcare_processed.csv"
)

print("Shape:", df.shape)

df.head()

Shape: (110527, 18)


,Gender,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No_show,WaitingDays,ScheduledHour,ScheduledDayOfWeek,AppointmentDayOfWeek,AppointmentMonth,AgeGroup,WaitingGroup,ScheduledTimeGroup
0,F,62.0,JARDIM DA PENHA,0,1,0,0,0,0,0,0.0,18,Friday,Friday,4,Middle age,Same Day,Late Evening
1,M,56.0,JARDIM DA PENHA,0,0,0,0,0,0,0,0.0,16,Friday,Friday,4,Adult,Same Day,Evening
2,F,62.0,MATA DA PRAIA,0,0,0,0,0,0,0,0.0,16,Friday,Friday,4,Middle age,Same Day,Evening
3,F,8.0,PONTAL DE CAMBURI,0,0,0,0,0,0,0,0.0,17,Friday,Friday,4,Child,Same Day,Evening
4,F,56.0,JARDIM DA PENHA,0,1,1,0,0,0,0,0.0,16,Friday,Friday,4,Adult,Same Day,Evening


In [4]:
# Separate X and y

In [5]:
X = df.drop(columns=["No_show"])
y = df["No_show"]

In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (110527, 17)
y shape: (110527,)


In [7]:
# Identify feature types

In [8]:
numeric_features = [
    "Age",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received",
    "WaitingDays",
    "ScheduledHour",
    "AppointmentMonth"
]

In [9]:
categorical_features = [
    "Gender",
    "Neighbourhood",
    "ScheduledDayOfWeek",
    "AppointmentDayOfWeek",
    "AgeGroup",
    "WaitingGroup",
    "ScheduledTimeGroup"
]

In [10]:
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

Numeric features: 10
Categorical features: 7
Total features: 17


In [11]:
# Train/Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [13]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

X_train: (88421, 17)
X_test: (22106, 17)

Training target distribution:
No_show
0    0.798068
1    0.201932
Name: proportion, dtype: float64

Testing target distribution:
No_show
0    0.798064
1    0.201936
Name: proportion, dtype: float64


In [14]:
# Build numerical preprocessing

In [15]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [16]:
# Build categorical preprocessing

In [17]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [18]:
# Combine preprocessing

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [20]:
# Create Logistic Regression pipeline

In [21]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [22]:
# Train

In [23]:
logistic_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
# Make predictions

In [25]:
y_pred = logistic_pipeline.predict(X_test)

In [26]:
y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

In [27]:
#Evaluate the baseline

In [28]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.7974305618384149
Precision: 0.4453125
Recall   : 0.012768817204301076
F1 Score : 0.024825783972125436
ROC-AUC  : 0.7310007221484736


In [29]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


Classification Report:
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     17642
           1       0.45      0.01      0.02      4464

    accuracy                           0.80     22106
   macro avg       0.62      0.50      0.46     22106
weighted avg       0.73      0.80      0.71     22106



In [30]:
lr_pr_auc = average_precision_score(
    y_test,
    y_prob
)

print("Logistic Regression PR-AUC:", lr_pr_auc)

Logistic Regression PR-AUC: 0.35656119538693243


In [31]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


Confusion Matrix:
[[17571    71]
 [ 4407    57]]


In [32]:
# Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier

In [34]:
# Create Random Forest pipeline

In [35]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [36]:
# Train Random Forest

In [37]:
random_forest_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [38]:
# Predict

In [39]:
rf_pred = random_forest_pipeline.predict(X_test)

rf_prob = (
    random_forest_pipeline
    .predict_proba(X_test)[:, 1]
)

In [40]:
# Evaluate Random Forest

In [41]:
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, rf_prob))

Accuracy : 0.8001447570795259
Precision: 0.5168869309838473
Recall   : 0.15770609318996415
F1 Score : 0.24167524888431172
ROC-AUC  : 0.7448243126739342


In [42]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred
    )
)


Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.88     17642
           1       0.52      0.16      0.24      4464

    accuracy                           0.80     22106
   macro avg       0.67      0.56      0.56     22106
weighted avg       0.76      0.80      0.76     22106



In [43]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_pred
    )
)


Confusion Matrix:
[[16984   658]
 [ 3760   704]]


In [44]:
rf_pr_auc = average_precision_score(
    y_test,
    rf_prob
)

print("Random Forest PR-AUC:", rf_pr_auc)

Random Forest PR-AUC: 0.4074693312628283


In [45]:
# XGBoost

In [46]:
from xgboost import XGBClassifier

In [47]:
# Create the XGBoost pipeline

In [48]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [49]:
# Train

In [50]:
xgb_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
# Predict

In [52]:
xgb_pred = xgb_pipeline.predict(X_test)

xgb_prob = (
    xgb_pipeline
    .predict_proba(X_test)[:, 1]
)

In [53]:
# Evaluate

In [54]:
xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

xgb_precision = precision_score(
    y_test,
    xgb_pred
)

xgb_recall = recall_score(
    y_test,
    xgb_pred
)

xgb_f1 = f1_score(
    y_test,
    xgb_pred
)

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_prob
)

xgb_pr_auc = average_precision_score(
    y_test,
    xgb_prob
)

print("Accuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1 Score :", xgb_f1)
print("ROC-AUC  :", xgb_roc_auc)
print("PR-AUC   :", xgb_pr_auc)

Accuracy : 0.7980186374739889
Precision: 0.4943820224719101
Recall   : 0.00985663082437276
F1 Score : 0.019327915660004392
ROC-AUC  : 0.7414640976201708
PR-AUC   : 0.3742269042791907


In [55]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        xgb_pred
    )
)


Classification Report:
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     17642
           1       0.49      0.01      0.02      4464

    accuracy                           0.80     22106
   macro avg       0.65      0.50      0.45     22106
weighted avg       0.74      0.80      0.71     22106



In [56]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        xgb_pred
    )
)


Confusion Matrix:
[[17597    45]
 [ 4420    44]]


In [57]:
# Model Tuning

In [58]:
# Tune Random Forest

In [59]:
from sklearn.model_selection import GridSearchCV

In [60]:
rf_param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"]
}

In [61]:
rf_grid = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

In [62]:
rf_grid.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], ...}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [63]:
print("Best Parameters:")
print(rf_grid.best_params_)

Best Parameters:
{'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}


In [64]:
print("Best CV ROC-AUC:")
print(rf_grid.best_score_)

Best CV ROC-AUC:
0.7511862603450269


In [65]:
#Get the tuned Random Forest

In [66]:
tuned_rf = rf_grid.best_estimator_

In [67]:
tuned_rf_pred = tuned_rf.predict(X_test)

tuned_rf_prob = (
    tuned_rf.predict_proba(X_test)[:, 1]
)

In [68]:
print("Accuracy :", accuracy_score(y_test, tuned_rf_pred))
print("Precision:", precision_score(y_test, tuned_rf_pred))
print("Recall   :", recall_score(y_test, tuned_rf_pred))
print("F1 Score :", f1_score(y_test, tuned_rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, tuned_rf_prob))
print("PR-AUC   :", average_precision_score(
    y_test,
    tuned_rf_prob
))

Accuracy : 0.8016828010494889
Precision: 0.696078431372549
Recall   : 0.03181003584229391
F1 Score : 0.060839760068551844
ROC-AUC  : 0.7590397187755352
PR-AUC   : 0.4189456507716276


In [69]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        tuned_rf_pred
    )
)


Confusion Matrix:
[[17580    62]
 [ 4322   142]]


In [70]:
# Create the XGBoost parameter grid

In [71]:
xgb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [72]:
#Create GridSearchCV

In [73]:
xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

In [74]:
# train
xgb_grid.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'model__colsample_bytree': [0.8, 1.0], 'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [3, 4, ...], 'model__n_estimators': [100, 200], ...}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [75]:
print("Best Parameters:")
print(xgb_grid.best_params_)

print("\nBest CV ROC-AUC:")
print(xgb_grid.best_score_)

Best Parameters:
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.8}

Best CV ROC-AUC:
0.7419647556296153


In [76]:
# Evaluate the tuned XGBoost

In [77]:
tuned_xgb = xgb_grid.best_estimator_

tuned_xgb_pred = tuned_xgb.predict(X_test)

tuned_xgb_prob = (
    tuned_xgb.predict_proba(X_test)[:, 1]
)

In [78]:
print("Accuracy :", accuracy_score(y_test, tuned_xgb_pred))
print("Precision:", precision_score(y_test, tuned_xgb_pred))
print("Recall   :", recall_score(y_test, tuned_xgb_pred))
print("F1 Score :", f1_score(y_test, tuned_xgb_pred))
print("ROC-AUC  :", roc_auc_score(y_test, tuned_xgb_prob))
print("PR-AUC   :", average_precision_score(
    y_test,
    tuned_xgb_prob
))

Accuracy : 0.7990138423957297
Precision: 0.5304347826086957
Recall   : 0.04099462365591398
F1 Score : 0.07610729881472239
ROC-AUC  : 0.7478756858328062
PR-AUC   : 0.38694362352553513


In [79]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        tuned_xgb_pred
    )
)


Confusion Matrix:
[[17480   162]
 [ 4281   183]]


In [80]:
# Test multiple thresholds

In [81]:
thresholds = np.arange(
    0.10,
    0.51,
    0.05
)

threshold_results = []

for threshold in thresholds:

    pred = (
        tuned_rf_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            pred,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

,Threshold,Precision,Recall,F1
0,0.10,0.284065,0.953629,0.437738
1,0.15,0.291559,0.932348,0.444207
2,0.20,0.312238,0.851030,0.456858
3,0.25,0.349610,0.702509,0.466875
4,0.30,0.398089,0.513217,0.448380
5,0.35,0.444989,0.305332,0.362163
6,0.40,0.513069,0.153898,0.236774
7,0.45,0.620968,0.068996,0.124194
8,0.50,0.696078,0.031810,0.060840


In [82]:
best_f1_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_f1_row

Threshold    0.250000
Precision    0.349610
Recall       0.702509
F1           0.466875
Name: 3, dtype: float64

In [83]:
fine_thresholds = np.arange(
    0.10,
    0.501,
    0.01
)

fine_results = []

for threshold in fine_thresholds:

    pred = (
        tuned_rf_prob >= threshold
    ).astype(int)

    fine_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            pred,
            zero_division=0
        )
    })

fine_threshold_df = pd.DataFrame(
    fine_results
)

fine_threshold_df.head()

,Threshold,Precision,Recall,F1
0,0.10,0.284065,0.953629,0.437738
1,0.11,0.285090,0.950493,0.438621
2,0.12,0.286391,0.947581,0.439846
3,0.13,0.287753,0.944220,0.441084
4,0.14,0.288959,0.937500,0.441759


In [84]:
fine_threshold_df.loc[
    fine_threshold_df["F1"].idxmax()
]

Threshold    0.260000
Precision    0.360116
Recall       0.668683
F1           0.468125
Name: 16, dtype: float64

In [85]:
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

In [86]:
X_train_final, X_valid, y_train_final, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.111111,
    random_state=42,
    stratify=y_train_full
)

In [87]:
print("Training:", X_train_final.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test_final.shape)

Training: (88421, 17)
Validation: (11053, 17)
Test: (11053, 17)


In [88]:
print(rf_grid.best_params_)

{'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}


In [89]:
final_rf = rf_grid.best_estimator_

final_rf.fit(
    X_train_final,
    y_train_final
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [90]:
# Get validation probabilities

In [91]:
valid_prob = final_rf.predict_proba(
    X_valid
)[:, 1]

In [92]:
# best validation threshold

In [93]:
thresholds = np.arange(
    0.10,
    0.51,
    0.01
)

validation_results = []

for threshold in thresholds:

    valid_pred = (
        valid_prob >= threshold
    ).astype(int)

    validation_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(
            y_valid,
            valid_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            valid_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            valid_pred,
            zero_division=0
        )
    })

validation_threshold_df = pd.DataFrame(
    validation_results
)

In [94]:
best_validation_threshold = (
    validation_threshold_df.loc[
        validation_threshold_df["F1"].idxmax()
    ]
)

best_validation_threshold

Threshold    0.250000
Precision    0.341755
Recall       0.690860
F1           0.457295
Name: 15, dtype: float64

In [95]:
# Final test evaluation

In [96]:
final_test_prob = final_rf.predict_proba(
    X_test_final
)[:, 1]

In [97]:
selected_threshold = best_validation_threshold[
    "Threshold"
]

In [98]:
final_test_pred = (
    final_test_prob >= selected_threshold
).astype(int)

In [99]:
print("Selected Threshold:", selected_threshold)

print(
    "Accuracy :",
    accuracy_score(y_test_final, final_test_pred)
)

print(
    "Precision:",
    precision_score(
        y_test_final,
        final_test_pred
    )
)

print(
    "Recall   :",
    recall_score(
        y_test_final,
        final_test_pred
    )
)

print(
    "F1 Score :",
    f1_score(
        y_test_final,
        final_test_pred
    )
)

print(
    "ROC-AUC  :",
    roc_auc_score(
        y_test_final,
        final_test_prob
    )
)

print(
    "PR-AUC   :",
    average_precision_score(
        y_test_final,
        final_test_prob
    )
)

Selected Threshold: 0.25
Accuracy : 0.6727585270967158
Precision: 0.3471639814610461
Recall   : 0.7047491039426523
F1 Score : 0.46517817536596184
ROC-AUC  : 0.759260444385933
PR-AUC   : 0.4265584595537453


In [100]:
print(
    confusion_matrix(
        y_test_final,
        final_test_pred
    )
)

[[5863 2958]
 [ 659 1573]]
